Classification baseline: TF-IDF features and logistic regression.



Evaluation: Class-specific errors, confusion matrix, precision, recall, and F1.



Uncertainty: Test a rule for sending low-confidence predictions to human review.



Clustering: Group requests from deliberately held-out categories to simulate unfamiliar intents. Evaluate whether the groups reveal meaningful patterns.



Engineering: Build an API, test response latency, and compare your baseline with an embedding-based approach.


initial business objective is:
Develop a system that classifies incoming banking-support requests by customer intent, helping route them to the appropriate support workflow.

In [1]:
import json
import pandas as pd
with open("../data/raw/categories.json") as f:
    categories = json.load(f)
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

In [2]:
print(type(categories))
print(len(categories))
print(categories[:5])

<class 'list'>
77
['card_arrival', 'card_linking', 'exchange_rate', 'card_payment_wrong_exchange_rate', 'extra_charge_on_statement']


In [23]:
train_df.head(30)

,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival
5,When did you send me my new card?,card_arrival
6,Do you have info about the card on delivery?,card_arrival
7,What do I do if I still have not received my n...,card_arrival
8,Does the package with my card have tracking?,card_arrival
9,I ordered my card but it still isn't here,card_arrival


In [5]:
train_df.shape

(10003, 2)

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   text      10003 non-null  str  
 1   category  10003 non-null  str  
dtypes: str(2)
memory usage: 156.4 KB


In [7]:
train_df.describe()

,text,category
count,10003,10003
unique,10003,77
top,I am still waiting on my card?,card_payment_fee_charged
freq,1,187


In [13]:
train_df.isna().sum()

text        0
category    0
dtype: int64

In [18]:
train_df["category"].value_counts()

category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64

In [21]:
train_df.duplicated().sum()

np.int64(0)

TEXT CLEANING

In [24]:
from spellchecker import SpellChecker


# Select the text column
text_review = train_df["text"]

# Extract individual words from all messages
words = (
    text_review
    .fillna("")
    .str.lower()
    .str.findall(r"[a-z]+(?:'[a-z]+)?")
    .explode()
)

# Count unique words and their frequencies
word_counts = words.value_counts()

# Initialize spell checker
spell = SpellChecker(distance=1)

# Flag unrecognized words
flagged_words = spell.unknown(word_counts.index.tolist())

# Create a review table
spelling_review = pd.DataFrame({
    "word": list(flagged_words)
})

spelling_review["frequency"] = spelling_review["word"].map(word_counts)

# Show the most frequently occurring flagged words
spelling_review = spelling_review.sort_values(
    "frequency", ascending=False
)

display(spelling_review)

,word,frequency
123,atm,474
39,app,469
50,cheque,105
106,gbp,66
88,passcode,64
...,...,...
143,roder,1
144,dp,1
145,cardsper,1
146,cryptocurency,1


In [25]:
import re

words_to_check = {
    "Critical banking terms": [
        "recieve", "recieving", "recived",
        "recepient", "reciepient",
        "transfered", "tranferred",
        "tranfering", "tranferring",
        "transger", "withdrawl",
        "witdrawal", "withdrawed",
        "transation", "tranasaction",
        "paymet", "acount"
    ],

    "Negation and contractions": [
        "didnt", "dont", "isnt",
        "hasnt", "havnt", "doesnt",
        "couldnt", "wouldnt", "shouldnt",
        "wasnt", "din't", "havn't"
    ],

    "Ambiguous terms": [
        "stillpending", "gpb",
        "currancy", "currecy",
        "beneficiery", "wd", "dp",
        "tup", "refurn", "traners",
        "roder", "cardsper"
    ]
}

summary = []
examples = []

for group, terms in words_to_check.items():
    for word in terms:

        pattern = rf"(?<!\w){re.escape(word)}(?!\w)"

        matches = train_df.loc[
            train_df["text"].str.contains(
                pattern, case=False, na=False, regex=True
            ),
            ["text", "category"]
        ]

        summary.append({
            "group": group,
            "word": word,
            "messages_found": len(matches)
        })

        for _, row in matches.head(5).iterrows():
            examples.append({
                "group": group,
                "flagged_word": word,
                "original_message": row["text"],
                "intent": row["category"]
            })

summary_df = pd.DataFrame(summary)
examples_df = pd.DataFrame(examples)

display(summary_df.sort_values(
    "messages_found", ascending=False
))

display(examples_df)

,group,word,messages_found
17,Negation and contractions,didnt,22
18,Negation and contractions,dont,16
10,Critical banking terms,withdrawl,11
19,Negation and contractions,isnt,6
33,Ambiguous terms,beneficiery,4
5,Critical banking terms,transfered,4
0,Critical banking terms,recieve,4
34,Ambiguous terms,wd,4
20,Negation and contractions,hasnt,3
23,Negation and contractions,couldnt,3


,group,flagged_word,original_message,intent
0,Critical banking terms,recieve,When will I recieve my new card?,card_arrival
1,Critical banking terms,recieve,What's the expected wait time to recieve my ne...,card_arrival
2,Critical banking terms,recieve,How long does it take to recieve my new card?,card_arrival
3,Critical banking terms,recieve,How can I activate my new card? I didn't recie...,activate_my_card
4,Critical banking terms,recieving,What are the charges for recieving SEPA transf...,top_up_by_bank_transfer_charge
...,...,...,...,...
70,Ambiguous terms,dp,How dp i top up with a cheque ?,top_up_by_cash_or_cheque
71,Ambiguous terms,tup,Can an apple watch be used to tup-up?,apple_pay_or_google_pay
72,Ambiguous terms,refurn,How long after a failed topup will funds refur...,pending_top_up
73,Ambiguous terms,roder,Can you point me in the right direction of whe...,get_disposable_virtual_card


In [46]:
import re

corrections = {
    
    "recieve": "receive",
    "recieving": "receiving",
    "recived": "received",
    "recepient": "recipient",
    "reciepient": "recipient",

   
    "transfered": "transferred",
    "tranferred": "transferred",
    "tranfering": "transferring",
    "tranferring": "transferring",

    
    "withdrawl": "withdrawal",
    "witdrawal": "withdrawal",
    "transation": "transaction",
    "tranasaction": "transaction",
    "paymet": "payment",
    "acount": "account",
    "beneficiery": "beneficiary",

    
    "stillpending": "still pending",
    "gpb": "gbp",
    "currancy": "currency",
    "currecy": "currency",
    "adress": "address",
    "wd": "withdrawl",
     "dp": "do",
      "tup":"top",
      "refurn":"return",
       "roder":"order",
       "cardsper":"cards per",
       "redepoised":"redeposited"
}
def correct_spelling(text):
    for wrong, correct in corrections.items():
        text = re.sub(
            rf"\b{re.escape(wrong)}\b",
            correct,
            text,
            flags=re.IGNORECASE
        )
    return text

train_df["text"] = train_df["text"].apply(correct_spelling)

In [48]:
train_df.loc[train_df["text"].str.contains("cards per", case=False, na=False)]

,text,category,text_clean
4355,"I need multiple cards per day, and they all ha...",disposable_card_limits,"I need multiple cards per day, and they all ha..."
4367,I need to make several disposable cards per day.,disposable_card_limits,I need to make several disposable cards per day.
4374,I need several cards per day. And I need to be...,disposable_card_limits,I need several cards per day. And I need to be...
4381,I have to make several disposable cards per day.,disposable_card_limits,I have to make several disposable cards per day.
4389,Can i make several disposable cards per day?,disposable_card_limits,Can i make several disposable cards per day?
4402,How many disposable cards per person?,disposable_card_limits,How many disposable cards per person?
4440,Is there a limit to the number of disposable c...,disposable_card_limits,Is there a limit to the number of disposable c...
4443,I must make several disposable cards per day.,disposable_card_limits,I must make several disposable cards per day.
4446,I have to make more then a few disposable card...,disposable_card_limits,I have to make more then a few disposable card...
